LOAD EMBEDDING MODEL

In [ ]:
from sentence_transformers import SentenceTransformer

# Model download & load
model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Model successfully loaded!")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1044.64it/s]


✅ Model successfully loaded!
Embedding dimension: 384


C:\Users\Haris\AppData\Local\Temp\ipykernel_9312\1238150387.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


DEFINE RESUME AND JOB SAMPLE

In [8]:
# Sample Resume text (sections/bullet points)
resume_text = """
Proficient in Python, SQL, and data analysis using Pandas and NumPy.
Experienced in training machine learning models using Scikit-Learn and PyTorch.
Built and evaluated deep learning models including LSTMs for time-series forecasting.
Developed web applications and dashboards using Streamlit and FastAPI.
Familiar with Git, GitHub version control, and collaborative repository workflows.
"""

# Sample Job Description
jd_text = """
Looking for a Machine Learning Engineer with 2+ years of experience.
Must have strong hands-on experience with Python and SQL.
Experience in deep learning frameworks such as TensorFlow or PyTorch.
Solid understanding of containerization using Docker and Kubernetes.
Familiarity with cloud platforms like AWS or Google Cloud.
Experience building interactive dashboards or REST APIs using FastAPI.
"""

print(f"Resume characters: {len(resume_text)}")
print(f"JD characters: {len(jd_text)}")

Resume characters: 390
JD characters: 397


Text Preprocessing (Sentences / Requirements Split)

In [9]:
# Lines split To clean it 
resume_lines = [line.strip("- *• \t\n") for line in resume_text.strip().split("\n") if line.strip()]
jd_lines = [line.strip("- *• \t\n") for line in jd_text.strip().split("\n") if line.strip()]

print(f"Total Resume Points: {len(resume_lines)}")
print(f"Total JD Requirements: {len(jd_lines)}")
print("\nFirst JD Requirement:", jd_lines[0])

Total Resume Points: 5
Total JD Requirements: 6

First JD Requirement: Looking for a Machine Learning Engineer with 2+ years of experience.


Sentence-by-Sentence Embeddings aur Best Match Finder

In [ ]:
from sentence_transformers.util import cos_sim
import numpy as np

# Embeddings generation
resume_embeddings = model.encode(resume_lines, convert_to_tensor=True)
jd_embeddings = model.encode(jd_lines, convert_to_tensor=True)

# Similarity matrix (JD lines x Resume lines)
similarity_matrix = cos_sim(jd_embeddings, resume_embeddings).cpu().numpy()

results = []
for i, jd_req in enumerate(jd_lines):
    # find best match and index for each jb requirement
    best_match_idx = int(np.argmax(similarity_matrix[i]))
    best_score = float(similarity_matrix[i][best_match_idx])
    
    results.append({
        "jd_requirement": jd_req,
        "best_matching_resume_line": resume_lines[best_match_idx],
        "similarity_score": round(best_score * 100, 2)
    })

# Formatted view
import pandas as pd
df_results = pd.DataFrame(results)
df_results

,jd_requirement,best_matching_resume_line,similarity_score
0,Looking for a Machine Learning Engineer with 2...,Experienced in training machine learning model...,37.74
1,Must have strong hands-on experience with Pyth...,"Proficient in Python, SQL, and data analysis u...",71.74
2,Experience in deep learning frameworks such as...,Experienced in training machine learning model...,62.89
3,Solid understanding of containerization using ...,"Familiar with Git, GitHub version control, and...",22.88
4,Familiarity with cloud platforms like AWS or G...,"Familiar with Git, GitHub version control, and...",34.29
5,Experience building interactive dashboards or ...,Developed web applications and dashboards usin...,73.01


Overall Score aur Gaps Identification

In [ ]:
# consider gap for score that is below 50%   (for semantic similarity)
THRESHOLD = 50.0

matched_skills = df_results[df_results['similarity_score'] >= THRESHOLD]
missing_gaps = df_results[df_results['similarity_score'] < THRESHOLD]

overall_match_score = round(df_results['similarity_score'].mean(), 2)

print(f"🎯 Overall Resume vs JD Match Score: {overall_match_score}%\n")
print(f"✅ Matched Requirements ({len(matched_skills)}/{len(df_results)}):")
for _, row in matched_skills.iterrows():
    print(f"  • {row['jd_requirement']} (Score: {row['similarity_score']}%)")

print(f"\n⚠️ Missing / Gap Requirements ({len(missing_gaps)}/{len(df_results)}):")
for _, row in missing_gaps.iterrows():
    print(f"  • {row['jd_requirement']} (Score: {row['similarity_score']}%)")

🎯 Overall Resume vs JD Match Score: 50.42%

✅ Matched Requirements (3/6):
  • Must have strong hands-on experience with Python and SQL. (Score: 71.74%)
  • Experience in deep learning frameworks such as TensorFlow or PyTorch. (Score: 62.89%)
  • Experience building interactive dashboards or REST APIs using FastAPI. (Score: 73.01%)

⚠️ Missing / Gap Requirements (3/6):
  • Looking for a Machine Learning Engineer with 2+ years of experience. (Score: 37.74%)
  • Solid understanding of containerization using Docker and Kubernetes. (Score: 22.88%)
  • Familiarity with cloud platforms like AWS or Google Cloud. (Score: 34.29%)


LLM Client Setup & API Key

In [ ]:
import os
from groq import Groq

# Paste your Actual APi key here 
GROQ_API_KEY = "YOUR_API_KEY_HERE" 

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialized!")

✅ Groq client initialized!


Structured JSON Prompt & Execution

In [ ]:
import json

# format Pre-computed gaps and matches to clean bullet points
matched_list = "\n".join([f"- {row['jd_requirement']} (Match: {row['similarity_score']}%)" for _, row in matched_skills.iterrows()])
gaps_list = "\n".join([f"- {row['jd_requirement']} (Match: {row['similarity_score']}%)" for _, row in missing_gaps.iterrows()])

system_prompt = """
You are an expert technical recruiter and ATS specialist.
Analyze the provided candidate match evaluation and return your output strictly as valid JSON.
Do not wrap in markdown quotes like ```json. Return pure JSON only.

JSON Schema:
{
  "summary": "1-2 sentence overall evaluation",
  "key_strengths": ["list of identified strengths"],
  "critical_gaps": ["list of critical missing areas to address"],
  "actionable_resume_bullets": ["2-3 specific, tailored bullet points candidate can add to resume to bridge the gaps honestly"]
}
"""

user_prompt = f"""
Candidate Overall Match Score: {overall_match_score}%

Matched Requirements:
{matched_list}

Identified Gaps:
{gaps_list}

Resume Content:
{resume_text}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

# Parse response
feedback_data = json.loads(response.choices[0].message.content)
print(json.dumps(feedback_data, indent=2))

{
  "summary": "The candidate demonstrates solid Python, SQL, and deep\u2011learning expertise with FastAPI experience, but lacks the required 2+ years of ML engineering experience, containerization, and cloud platform exposure.",
  "key_strengths": [
    "Strong hands\u2011on Python and SQL skills",
    "Experience with deep learning frameworks (PyTorch)",
    "Built interactive dashboards and REST APIs using FastAPI"
  ],
  "critical_gaps": [
    "Less than 2 years of dedicated Machine Learning Engineer experience",
    "Limited or no experience with Docker and Kubernetes containerization",
    "Minimal exposure to cloud platforms such as AWS or Google Cloud"
  ],
  "actionable_resume_bullets": [
    "Developed and containerized a FastAPI inference service using Docker, and deployed it on AWS Elastic Beanstalk (or GCP Cloud Run) to demonstrate cloud deployment capabilities",
    "Implemented CI/CD pipelines with GitHub Actions to build Docker images and orchestrate Kubernetes deploym

Clean Formatted Display

In [18]:
print(f"📋 ATS EVALUATION REPORT ({overall_match_score}% Match)")
print("=" * 60)
print(f"\n📝 SUMMARY:\n{feedback_data.get('summary')}")

print("\n🌟 KEY STRENGTHS:")
for s in feedback_data.get("key_strengths", []):
    print(f"  • {s}")

print("\n⚠️ CRITICAL GAPS:")
for g in feedback_data.get("critical_gaps", []):
    print(f"  • {g}")

print("\n💡 RECOMMENDED RESUME IMPROVEMENTS (Actionable Bullets):")
for b in feedback_data.get("actionable_resume_bullets", []):
    print(f"  ➜ {b}")

📋 ATS EVALUATION REPORT (50.42% Match)

📝 SUMMARY:
The candidate demonstrates solid Python, SQL, and deep‑learning expertise with FastAPI experience, but lacks the required 2+ years of ML engineering experience, containerization, and cloud platform exposure.

🌟 KEY STRENGTHS:
  • Strong hands‑on Python and SQL skills
  • Experience with deep learning frameworks (PyTorch)
  • Built interactive dashboards and REST APIs using FastAPI

⚠️ CRITICAL GAPS:
  • Less than 2 years of dedicated Machine Learning Engineer experience
  • Limited or no experience with Docker and Kubernetes containerization
  • Minimal exposure to cloud platforms such as AWS or Google Cloud

💡 RECOMMENDED RESUME IMPROVEMENTS (Actionable Bullets):
  ➜ Developed and containerized a FastAPI inference service using Docker, and deployed it on AWS Elastic Beanstalk (or GCP Cloud Run) to demonstrate cloud deployment capabilities
  ➜ Implemented CI/CD pipelines with GitHub Actions to build Docker images and orchestrate Kubern